In [1]:
import pandas as pd
import requests
import os
from tqdm import tqdm

In [3]:
ctti_zip_path = "./downloads/CTTI_new.zip"
import zipfile

with zipfile.ZipFile(ctti_zip_path, 'r') as zip_ref:
    names = zip_ref.namelist()
    browse_conditions = pd.read_csv(zip_ref.open([n for n in names if 'browse_conditions.txt' in n][0]), sep='|')

# Normalize and deduplicate disease terms
browse_conditions['disease'] = browse_conditions['downcase_mesh_term'].str.strip().str.lower()
unique_diseases = browse_conditions['disease'].dropna().unique()

In [4]:
def get_icd_from_clinicaltables(disease):
    url = f"https://clinicaltables.nlm.nih.gov/api/icd10cm/v3/search?sf=code,name&terms={requests.utils.quote(disease)}"
    try:
        r = requests.get(url, timeout=5)
        data = r.json()
        codes = data[1]
        return codes if codes else None
    except Exception as e:
        return None

In [5]:
disease_icd_map = []
for disease in tqdm(unique_diseases):
    icds = get_icd_from_clinicaltables(disease)
    disease_icd_map.append({
        "disease": disease,
        "icd": icds if icds else ['None'],
        "count": browse_conditions[browse_conditions['disease'] == disease].shape[0]
    })


100%|████████████████████████████████████████████████████████████████████████████| 4489/4489 [1:05:39<00:00,  1.14it/s]


In [6]:
output_df = pd.DataFrame(disease_icd_map)
os.makedirs("data", exist_ok=True)
output_df.to_csv("data/diseases.csv", index=False)
print("Saved to data/diseases.csv")  

Saved to data/diseases.csv
